# Gradient Boosting: XGBoost, LightGBM, CatBoost

Gradient boosting is the most widely used ML algorithm in industry for structured/tabular data.

1. **Boosting Concept** - Sequential ensemble that learns from errors
2. **XGBoost** - The original powerhouse
3. **LightGBM** - Faster training with histogram-based splits
4. **CatBoost** - Native categorical feature handling
5. **Hyperparameter Tuning with Optuna**

**Dataset**: Breast Cancer (binary classification)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

sns.set_theme(style="whitegrid")

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 1. How Gradient Boosting Works

Unlike Random Forests (parallel, independent trees), boosting builds trees **sequentially**:

1. Fit a simple model to the data
2. Compute residuals (errors)
3. Fit the next model to predict the residuals
4. Add the new model's predictions (scaled by learning rate) to the ensemble
5. Repeat

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

where $\eta$ is the learning rate and $h_m$ is the new weak learner.

## 2. XGBoost

In [ ]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,     # L1 regularization
    reg_lambda=1.0,    # L2 regularization
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

xgb_clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_pred_xgb = xgb_clf.predict(X_test)
y_prob_xgb = xgb_clf.predict_proba(X_test)[:, 1]

print("XGBoost Results:")
print(f"  Accuracy: {xgb_clf.score(X_test, y_test):.4f}")
print(f"  ROC-AUC:  {roc_auc_score(y_test, y_prob_xgb):.4f}")

## 3. LightGBM

In [ ]:
lgb_clf = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

lgb_clf.fit(X_train, y_train, eval_set=[(X_test, y_test)])

y_prob_lgb = lgb_clf.predict_proba(X_test)[:, 1]
print(f"\nLightGBM Results:")
print(f"  Accuracy: {lgb_clf.score(X_test, y_test):.4f}")
print(f"  ROC-AUC:  {roc_auc_score(y_test, y_prob_lgb):.4f}")

## 4. CatBoost

In [ ]:
cat_clf = CatBoostClassifier(
    iterations=200,
    depth=4,
    learning_rate=0.1,
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=0,
)

cat_clf.fit(X_train, y_train, eval_set=(X_test, y_test))

y_prob_cat = cat_clf.predict_proba(X_test)[:, 1]
print(f"CatBoost Results:")
print(f"  Accuracy: {cat_clf.score(X_test, y_test):.4f}")
print(f"  ROC-AUC:  {roc_auc_score(y_test, y_prob_cat):.4f}")

## 5. Comparison

In [ ]:
# Compare all three with cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "XGBoost": xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42, eval_metric="logloss", n_jobs=-1),
    "LightGBM": lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42, verbose=-1, n_jobs=-1),
    "CatBoost": CatBoostClassifier(iterations=200, depth=4, learning_rate=0.1, random_seed=42, verbose=0),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring="roc_auc")
    results[name] = scores
    print(f"{name:12s}: ROC-AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")

# Boxplot comparison
plt.figure(figsize=(8, 5))
plt.boxplot(results.values(), labels=results.keys())
plt.ylabel("ROC-AUC")
plt.title("Gradient Boosting Model Comparison (5-Fold CV)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, model) in zip(axes, [("XGBoost", xgb_clf), ("LightGBM", lgb_clf), ("CatBoost", cat_clf)]):
    importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True).tail(10)
    importance.plot(kind="barh", ax=ax, color="teal")
    ax.set_title(f"{name} - Top 10 Features")

plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning with Optuna

Optuna uses Bayesian optimization - much more efficient than grid search.

In [ ]:
# Uncomment and run if optuna is installed:
# pip install optuna

# import optuna
# optuna.logging.set_verbosity(optuna.logging.WARNING)
#
# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 500),
#         "max_depth": trial.suggest_int("max_depth", 3, 8),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
#         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
#         "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
#         "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
#     }
#     model = lgb.LGBMClassifier(**params, random_state=42, verbose=-1, n_jobs=-1)
#     scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="roc_auc")
#     return scores.mean()
#
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=50)
#
# print(f"Best ROC-AUC: {study.best_value:.4f}")
# print(f"Best params: {study.best_params}")

## When to Use Which

| Library | Best For |
|---------|----------|
| **XGBoost** | General purpose, well-documented, strong Kaggle track record |
| **LightGBM** | Large datasets (fastest), high-cardinality categoricals |
| **CatBoost** | Categorical features without manual encoding, out-of-the-box performance |

## Key Takeaways

1. **Gradient boosting dominates tabular data** - it's the go-to for structured problems
2. **Learning rate and n_estimators are coupled** - lower LR needs more trees
3. **Early stopping prevents overfitting** - use eval_set with patience
4. **Use Optuna/Bayesian optimization** over grid search for hyperparameter tuning
5. **Subsample and colsample add regularization** through randomness (similar to RF)